In [2]:
import sys
from pathlib import Path

API = Path.cwd().parent                     # .../StaffScheduling/api (Jupyter startet im Ordner des Notebooks)
ROOT = API.parent
sys.path[:0] = [str(API), str(API / "tests")]   # app.* und die Test-Bausteine importierbar machen

# --- Datenstrukturen (nur app.domain: braucht pydantic + holidays, keine Datenbank, kein OR-Tools) ---
from app.domain import (
    POLICY,                 # Regelparameter: min_rest_minutes, balance_tolerance_minutes, max_consecutive_nights, ...
    SchedulingDataset,      # die komplette Eingabe eines Monats
    Assignment,             # ein Dienst (Ergebnis): employee_id, date, planning_unit_id, shift_id, staff_level
    Gap,                    # eine Lücke (Ergebnis)
    DemandRequirement, Shift, ShiftType, StaffLevel, AvailabilityType, WishType, FREE_WISHES,
    duty_times,             # echte Start-/Endzeiten eines Dienstes (inkl. Sommerzeit)
    is_working_day,         # Werktag ja/nein
    check_schedule,         # unabhängiger Schedule-Check: Regeln + Kennzahlen
)
from app.domain.rules import BLOCKING_AVAILABILITY, APPROVED_FREE
from app.domain.planning_unit import home_unit_id

# --- Testdaten laden ---
from app.solver.bundle import ScheduleInput     # liest input.json der Beispielmonate
import scheduling as fx                         # api/tests/scheduling.py: Bausteine für kleine Testfälle


In [3]:
# Variante A: ein echter Monat (7 Stationen, ~96 Mitarbeitende, Kontext, Konten, Abwesenheiten)
dataset = ScheduleInput.model_validate_json((ROOT / "plaene" / "2026-03" / "input.json").read_text()).dataset

In [4]:
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

def table(rows, drop=()):
    """Eine Liste von Pydantic-Objekten als DataFrame."""
    return pd.DataFrame([row.model_dump(mode="json", exclude=set(drop)) for row in rows])

print(f"Monat: {dataset.planning_month.label} ({len(dataset.planning_month.dates)} Tage)")
print(f"Stationen: {sorted(dataset.station_ids)}")
for name in ["planning_units", "shifts", "employees", "planning_unit_memberships", "demand_requirements",
             "availability", "wishes", "monthly_work_accounts"]:
    print(f"{name:28s} {len(getattr(dataset, name)):5d} Zeilen")
print(f"{'context.duties':28s} {len(dataset.context.duties):5d} Zeilen  ({dataset.context.covered_from} bis {dataset.context.covered_until})")


Monat: 2026-03 (31 Tage)
Stationen: [77, 78, 79, 83, 85, 88, 337]
planning_units                   9 Zeilen
shifts                           4 Zeilen
employees                       96 Zeilen
planning_unit_memberships      205 Zeilen
demand_requirements           1142 Zeilen
availability                    46 Zeilen
wishes                           0 Zeilen
monthly_work_accounts           96 Zeilen
context.duties                 353 Zeilen  (2026-02-24 bis 2026-03-31)


In [6]:
display(table(dataset.planning_units))

def hhmm(minute):
    return f"{minute // 60 % 24:02d}:{minute % 60:02d}" + (" (+1)" if minute >= 1440 else "")

shifts = pd.DataFrame([
    {"shift_id": s.shift_id, "code": s.code, "type": s.type.value,
     "start": hhmm(s.start_minute), "end": hhmm(s.end_minute),
     "segments": ", ".join(f"{hhmm(g.start_minute)}-{hhmm(g.end_minute)}" for g in s.segments),
     "net_work_minutes": s.net_work_minutes}
    for s in dataset.shifts
])
display(shifts)


,planning_unit_id,display_name,type
0,68,PE 68,jumper_pool
1,77,PE 77,station
2,78,PE 78,station
3,79,PE 79,station
4,83,PE 83,station
5,85,PE 85,station
6,88,PE 88,station
7,337,PE 337,station
8,408,PE 408,jumper_pool


,shift_id,code,type,start,end,segments,net_work_minutes
0,1113,F,early,05:55,13:25,"05:55-10:00, 10:30-13:25",420
1,1453,Z,intermediate,08:30,14:15,08:30-14:15,345
2,1605,S,late,13:15,21:00,"13:15-16:00, 16:30-21:00",435
3,1690,N,night,20:10,06:10 (+1),"20:10-21:45, 22:00-00:00 (+1), 00:30 (+1)-06:1...",555


In [7]:
employees = table(dataset.employees)
display(employees)
display(employees["staff_level"].value_counts())

memberships = table(dataset.planning_unit_memberships)
display(memberships)
# Wer darf auf welcher Station mit welcher Qualifikation arbeiten?
display(pd.crosstab(memberships["employee_id"], memberships["planning_unit_id"]))


,employee_id,display_name,staff_level
0,1226,Allan Diethard,professional
1,918,Ambriz Waltraud,professional
2,1191,Anoch Theobald,professional
3,7378,Boulware Ina,professional
4,791,Branz Janett,professional
...,...,...,...
91,7490,Waldinger Annemargr,professional
92,7919,Weathers Irma,professional
93,818,Wülfroth Luitgart,professional
94,928,Wunderlich Daniele,professional


staff_level
professional    76
assistant       13
trainee          6
mfa              1
Name: count, dtype: int64

,planning_unit_id,employee_id,valid_from,valid_until,staff_level,is_home,is_replacement
0,79,1226,2022-09-01,2199-12-31,professional,True,False
1,68,918,2024-10-29,2199-12-31,professional,True,False
2,78,918,2025-12-01,2026-07-31,professional,False,True
3,83,918,2025-12-01,2026-07-31,professional,False,True
4,85,918,2025-12-01,2026-07-31,professional,False,True
...,...,...,...,...,...,...,...
200,337,7490,2025-12-01,2026-07-31,professional,False,True
201,77,7919,2024-10-01,2199-12-31,professional,True,False
202,79,818,2021-11-01,2199-12-31,professional,True,False
203,77,928,2022-05-01,2199-12-31,professional,True,False


planning_unit_id,68,77,78,79,83,85,88,337,408
employee_id,,,,,,,,,
459,0,0,0,0,0,1,0,0,0
790,0,1,0,0,0,0,0,0,0
791,0,1,0,0,0,0,0,0,0
803,0,0,0,0,0,1,0,0,0
818,0,0,0,1,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...
8048,0,1,1,1,1,1,1,1,1
8049,0,1,1,1,1,1,1,1,1
8050,0,1,1,1,1,1,1,1,1


In [8]:
demand = table(dataset.demand_requirements)
code = {s.shift_id: s.code for s in dataset.shifts}
demand["shift"] = demand["shift_id"].map(code)
display(demand.head(20))

# Raster je Station: Zeilen = Schicht und Qualifikation, Spalten = Tag
for unit, rows in demand.groupby("planning_unit_id"):
    print(f"Station {unit}")
    display(rows.pivot_table(index=["shift", "staff_level"], columns="date",
                             values="required_count", aggfunc="sum", fill_value=0))


,planning_unit_id,date,shift_id,staff_level,required_count,shift
0,77,2026-03-01,1113,assistant,2,F
1,77,2026-03-01,1113,professional,3,F
2,77,2026-03-01,1605,assistant,1,S
3,77,2026-03-01,1605,professional,3,S
4,77,2026-03-01,1690,professional,2,N
5,77,2026-03-02,1113,assistant,2,F
6,77,2026-03-02,1113,mfa,1,F
7,77,2026-03-02,1113,professional,3,F
8,77,2026-03-02,1113,trainee,1,F
9,77,2026-03-02,1453,professional,1,Z


Station 77


date                2026-03-01  2026-03-02  2026-03-03  2026-03-04  2026-03-05  2026-03-06  2026-03-07  2026-03-08  2026-03-09  2026-03-10  2026-03-11  2026-03-12  2026-03-13  2026-03-14  \
shift staff_level                                                                                                                                                                            
F     assistant              2           2           2           2           2           2           2           2           2           2           2           2           2           2   
      mfa                    0           1           1           1           1           1           0           0           1           1           1           1           1           0   
      professional           3           3           3           3           3           3           3           3           3           3           3           3           3           3   
      trainee                0           1           1           1           1           1           0           0           1           1           1           1           1           0   
N     professional           2           2           2           2           2           2           2           2           2           2           2           2           2           2   
S     assistant              1           2           2           2           2           2           1           1           2           2           2           2           2           1   
      professional           3           3           3           3           3           3           3           3           3           3           3           3           3           3   
Z     professional           0           1           1           1           1           1           0           0           1           1           1           1           1           0   

date                2026-03-15  2026-03-16  2026-03-17  2026-03-18  2026-03-19  2026-03-20  2026-03-21  2026-03-22  2026-03-23  2026-03-24  2026-03-25  2026-03-26  2026-03-27  2026-03-28  \
shift staff_level                                                                                                                                                                            
F     assistant              2           2           2           2           2           2           2           2           2           2           2           2           2           2   
      mfa                    0           1           1           1           1           1           0           0           1           1           1           1           1           0   
      professional           3           3           3           3           3           3           3           3           3           3           3           3           3           3   
      trainee                0           1           1           1           1           1           0           0           1           1           1           1           1           0   
N     professional           2           2           2           2           2           2           2           2           2           2           2           2           2           2   
S     assistant              1           2           2           2           2           2           1           1           2           2           2           2           2           1   
      professional           3           3           3           3           3           3           3           3           3           3           3           3           3           3   
Z     professional           0           1           1           1           1           1           0           0           1           1           1           1           1           0   

date                2026-03-29  2026-03-30  2026-03-31  
shift staff_level                                       
F     assistant              2           2           2  
      mfa                  

Station 78


date                2026-03-01  2026-03-02  2026-03-03  2026-03-04  2026-03-05  2026-03-06  2026-03-07  2026-03-08  2026-03-09  2026-03-10  2026-03-11  2026-03-12  2026-03-13  2026-03-14  \
shift staff_level                                                                                                                                                                            
F     assistant              0           1           1           1           1           1           0           0           1           1           1           1           1           0   
      professional           1           1           1           1           1           1           1           1           1           1           1           1           1           1   
      trainee                1           1           1           1           1           1           1           1           1           1           1           1           1           1   
N     professional           1           1           1           1           1           1           1           1           1           1           1           1           1           1   
S     professional           1           1           1           1           1           1           1           1           1           1           1           1           1           1   
      trainee                1           1           1           1           1           1           1           1           1           1           1           1           1           1   

date                2026-03-15  2026-03-16  2026-03-17  2026-03-18  2026-03-19  2026-03-20  2026-03-21  2026-03-22  2026-03-23  2026-03-24  2026-03-25  2026-03-26  2026-03-27  2026-03-28  \
shift staff_level                                                                                                                                                                            
F     assistant              0           1           1           1           1           1           0           0           1           1           1           1           1           0   
      professional           1           1           1           1           1           1           1           1           1           1           1           1           1           1   
      trainee                1           1           1           1           1           1           1           1           1           1           1           1           1           1   
N     professional           1           1           1           1           1           1           1           1           1           1           1           1           1           1   
S     professional           1           1           1           1           1           1           1           1           1           1           1           1           1           1   
      trainee                1           1           1           1           1           1           1           1           1           1           1           1           1           1   

date                2026-03-29  2026-03-30  2026-03-31  
shift staff_level                                       
F     assistant              0           1           1  
      professional           1           1           1  
      trainee                1           1           1  
N     professional           1           1           1  
S     professional           1           1           1  
      trainee                1           1           1

...

date                2026-03-01  2026-03-02  2026-03-03  2026-03-04  2026-03-05  2026-03-06  2026-03-07  2026-03-08  2026-03-09  2026-03-10  2026-03-11  2026-03-12  2026-03-13  2026-03-14  \
shift staff_level                                                                                                                                                                            
F     assistant              0           1           1           1           1           1           0           0           1           1           1           1           1           0   
      professional           2           2           2           2           2           2           2           2           2           2           2           2           2           2   
      trainee                1           1           1           1           1           1           1           1           1           1           1           1           1           1   
N     professional           1           1           1           1           1           1           1           1           1           1           1           1           1           1   
S     professional           2           2           2           2           2           2           2           2           2           2           2           2           2           2   
      trainee                1           1           1           1           1           1           1           1           1           1           1           1           1           1   
Z     professional           0           1           1           1           1           1           0           0           1           1           1           1           1           0   

date                2026-03-15  2026-03-16  2026-03-17  2026-03-18  2026-03-19  2026-03-20  2026-03-21  2026-03-22  2026-03-23  2026-03-24  2026-03-25  2026-03-26  2026-03-27  2026-03-28  \
shift staff_level                                                                                                                                                                            
F     assistant              0           1           1           1           1           1           0           0           1           1           1           1           1           0   
      professional           2           2           2           2           2           2           2           2           2           2           2           2           2           2   
      trainee                1           1           1           1           1           1           1           1           1           1           1           1           1           1   
N     professional           1           1           1           1           1           1           1           1           1           1           1           1           1           1   
S     professional           2           2           2           2           2           2           2           2           2           2           2           2           2           2   
      trainee                1           1           1           1           1           1           1           1           1           1           1           1           1           1   
Z     professional           0           1           1           1           1           1           0           0           1           1           1           1           1           0   

date                2026-03-29  2026-03-30  2026-03-31  
shift staff_level                                       
F     assistant              0           1           1  
      professional           2           2           2  
      trainee                1           1           1  
N     professional           1           1           1  
S     professional           2           2           2  
      trainee                1           1           1  
Z     professional           0           1           1

Station 88


,date,2026-03-01,2026-03-02,2026-03-03,2026-03-04,2026-03-05,2026-03-06,2026-03-07,2026-03-08,2026-03-09,2026-03-10,2026-03-11,2026-03-12,2026-03-13,2026-03-14,2026-03-15,2026-03-16,2026-03-17,2026-03-18,2026-03-19,2026-03-20,2026-03-21,2026-03-22,2026-03-23,2026-03-24,2026-03-25,2026-03-26,2026-03-27,2026-03-28,2026-03-29,2026-03-30,2026-03-31
shift,staff_level,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
F,professional,4,5,5,5,5,5,4,4,5,5,5,5,5,4,4,5,5,5,5,5,4,4,5,5,5,5,5,4,4,5,5
N,professional,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4
S,professional,4,5,5,5,5,5,4,4,5,5,5,5,5,4,4,5,5,5,5,5,4,4,5,5,5,5,5,4,4,5,5
Z,professional,1,0,0,0,0,0,1,1,0,0,0,0,0,1,1,0,0,0,0,0,1,1,0,0,0,0,0,1,1,0,0


Station 337


date                2026-03-01  2026-03-02  2026-03-03  2026-03-04  2026-03-05  2026-03-06  2026-03-07  2026-03-08  2026-03-09  2026-03-10  2026-03-11  2026-03-12  2026-03-13  2026-03-14  \
shift staff_level                                                                                                                                                                            
F     professional           1           1           1           1           1           1           1           1           1           1           1           1           1           1   
      trainee                1           1           1           1           1           1           1           1           1           1           1           1           1           1   
N     professional           1           1           1           1           1           1           1           1           1           1           1           1           1           1   
S     professional           1           1           1           1           1           1           1           1           1           1           1           1           1           1   
      trainee                1           1           1           1           1           1           1           1           1           1           1           1           1           1   

date                2026-03-15  2026-03-16  2026-03-17  2026-03-18  2026-03-19  2026-03-20  2026-03-21  2026-03-22  2026-03-23  2026-03-24  2026-03-25  2026-03-26  2026-03-27  2026-03-28  \
shift staff_level                                                                                                                                                                            
F     professional           1           1           1           1           1           1           1           1           1           1           1           1           1           1   
      trainee                1           1           1           1           1           1           1           1           1           1           1           1           1           1   
N     professional           1           1           1           1           1           1           1           1           1           1           1           1           1           1   
S     professional           1           1           1           1           1           1           1           1           1           1           1           1           1           1   
      trainee                1           1           1           1           1           1           1           1           1           1           1           1           1           1   

date                2026-03-29  2026-03-30  2026-03-31  
shift staff_level                                       
F     professional           1           1           1  
      trainee                1           1           1  
N     professional           1           1           1  
S     professional           1           1           1  
      trainee                1           1           1

In [9]:
accounts = table(dataset.monthly_work_accounts, drop={"credit_details"})
accounts["target_h"] = (accounts["target_minutes"] / 60).round(1)
accounts["open_minutes"] = accounts["target_minutes"] - accounts["credited_minutes"]   # noch zu verplanen
display(accounts)

# Einzelne Gutschriften (Urlaub, Schule, ...)
credits = pd.DataFrame([{"employee_id": a.employee_id, **c.model_dump(mode="json")}
                        for a in dataset.monthly_work_accounts for c in a.credit_details])
display(credits)


,employee_id,target_minutes,actual_minutes,credited_minutes,target_h,open_minutes
0,1226,10164,NaN,0,169.4,10164
1,918,6600,NaN,0,110.0,6600
2,1191,10164,NaN,0,169.4,10164
3,7378,10164,NaN,0,169.4,10164
4,791,7623,NaN,0,127.0,7623
...,...,...,...,...,...,...
91,7490,9155,NaN,0,152.6,9155
92,7919,10164,NaN,0,169.4,10164
93,818,7623,NaN,1735,127.0,5888
94,928,6098,NaN,0,101.6,6098


,employee_id,date,minutes,kind,source
0,2932,2026-03-02,462,approved_absence,TimeOffice U absence
1,2932,2026-03-03,462,approved_absence,TimeOffice U absence
2,2932,2026-03-04,462,approved_absence,TimeOffice U absence
3,2932,2026-03-05,462,approved_absence,TimeOffice U absence
4,2932,2026-03-06,462,approved_absence,TimeOffice U absence
5,5367,2026-03-16,462,approved_absence,TimeOffice U absence
6,5367,2026-03-17,462,approved_absence,TimeOffice U absence
7,5367,2026-03-18,462,approved_absence,TimeOffice U absence
8,5367,2026-03-19,462,approved_absence,TimeOffice U absence
9,5367,2026-03-20,462,approved_absence,TimeOffice U absence


In [10]:
context = table(dataset.context.duties)
if not context.empty:
    context["shift"] = context["shift_id"].map(code)
    display(context.pivot_table(index="employee_id", columns="date", values="shift",
                                aggfunc="first", fill_value=""))
display(table(dataset.context.availability))


date,2026-02-24,2026-02-25,2026-02-26,2026-02-27,2026-02-28
employee_id,,,,,
459,F,F,S,,
790,,,,,S
791,,,Z,Z,N
803,,F,N,N,
818,Z,F,F,F,F
...,...,...,...,...,...
8048,F,S,S,S,S
8049,S,S,S,N,N
8050,S,S,S,S,N


""


In [11]:
display(pd.Series(POLICY.model_dump(), name="Wert").to_frame())


,Wert
balance_tolerance_minutes,460
min_rest_minutes,660
max_consecutive_nights,3
night_recovery_minutes,2880
max_daily_work_minutes,600
average_daily_work_minutes,480
short_break_after_minutes,360
short_break_minutes,30
long_break_after_minutes,540
long_break_minutes,45


In [12]:
import gurobipy as gp
from gurobipy import GRB

from app.domain import POLICY, Assignment, Gap, SchedulingDataset, check_schedule
from app.settings import Settings
from app.solver.models import RunConfiguration, Solution, SolutionStatus, StageReport


class GurobiSolverService:
    def __init__(self, settings: Settings) -> None:
        self._settings = settings          # Zeitlimit, Threads, Seed, Logging aus .env

    def solve(self, dataset: SchedulingDataset, timeout: float | None = None) -> Solution:
        timeout = timeout if timeout is not None else self._settings.solver_max_time_seconds
        with gp.Env() as env, gp.Model("schedule", env=env) as m:
            # 1. Variablen, Constraints, Ziele aus `dataset` bauen  (dein Notebook-Code)
            # 2. m.Params.TimeLimit = timeout; m.optimize()
            # 3. Lösung übersetzen:
            assignments = (...)             # tuple[Assignment, ...]: alle x mit Wert 1
            gaps = (...)                    # tuple[Gap, ...]: unbesetzte Bedarfszeilen
            return Solution(
                status=SolutionStatus.OPTIMAL,                # oder FEASIBLE / INFEASIBLE / UNKNOWN
                configuration=RunConfiguration(policy=POLICY, timeout_seconds=timeout,
                                               search_workers=None, random_seed=None),
                wall_time_seconds=m.Runtime,
                assignments=assignments,
                gaps=gaps,
                stages=(StageReport(name="gaps", status=SolutionStatus.OPTIMAL, value=..., best_bound=...),),
                check=check_schedule(dataset, assignments, gaps),
            )


ModuleNotFoundError: No module named 'pydantic_settings'